In [67]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import mediapipe as mp

### Các bộ phân được tách
head 0

shoulder 11, 12

elbow 13, 14

wrist 15, 16

hip 23, 24

knee 25, 26

ankle 27, 28

In [68]:
video_path = "test.mp4"
model_path = "pose_landmarker_lite.task"

In [69]:
KEYPOINTS = [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]

BODY_PARTS = {
    'head':     [0],
    'shoulder': [11, 12],
    'elbow':    [13, 14],
    'wrist':    [15, 16],
    'hip':      [23, 24],
    'knee':     [25, 26],
    'ankle':    [27, 28],
}

GROUP = {
    0: 'head',
    11: 'left shoulder', 12: 'right shoulder',
    13: 'left elbow', 14: 'right elbow',
    15: 'left wrist', 16: 'right wrist',
    23: 'left hip', 24: 'right hip',
    25: 'left knee', 26: 'right knee',
    27: 'left ankle', 28: 'right ankle'
}

In [70]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

base_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1)


cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

def draw_landmarks(frame, result):
    if not result.pose_landmarks:
        return frame
    
    frame_copy = frame.copy()
    (h, w) = frame_copy.shape[:2]

    point_color = (0, 0, 255)

    KEYPOINTS = [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]

    for idx in KEYPOINTS:
        landmark = result.pose_landmarks[0][idx]

        if landmark.presence < 0.5 or landmark.visibility < 0.5:
            continue
        
        px = int(landmark.x * w)
        py = int(landmark.y * h)

        cv2.circle(frame_copy, (px, py), 6, point_color, -1)

    return frame_copy

extracted_data = []
def extract_data(result, frame_idx):
    if not result.pose_world_landmarks:
        return
    
    data = result.pose_world_landmarks[0]

    for landmark_id, landmark in enumerate(data):
        if landmark_id not in KEYPOINTS:
            continue

        extracted_data.append({
            'frame': frame_idx,
            'part': GROUP[landmark_id],
            'part_idx': landmark_id,
            'x': landmark.x,
            'y': landmark.y,
            'z': landmark.z,
            'visibility': landmark.visibility,
            'presence': landmark.presence
        })



with PoseLandmarker.create_from_options(base_options) as landmarker:
    frame_idx = 0
    rows = []

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        timestamp_ms = int((frame_idx / fps) * 1000)
        image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        result = landmarker.detect_for_video(image, timestamp_ms)

        if result.pose_landmarks:
            for landmark_id, landmark in enumerate(result.pose_landmarks[0]):
                if landmark.presence > 0.5 and landmark.visibility > 0.5:
                    print(f"Frame {frame_idx} | Landmark #{landmark_id}\nx={landmark.x:.3f}\ny={landmark.y:.3f}\nz={landmark.z:.3f}\nvisibility={landmark.visibility}\npresence={landmark.presence}")
        
        drawed_frame = draw_landmarks(frame, result)

        cv2.imshow("Camera", drawed_frame)
        
        extract_data(result, frame_idx)
        
        frame_idx += 1

cap.release()
cv2.destroyAllWindows()

Frame 0 | Landmark #0
x=0.386
y=0.613
z=-1.910
visibility=0.999153733253479
presence=0.9992303848266602
Frame 0 | Landmark #1
x=0.435
y=0.563
z=-1.733
visibility=0.999256432056427
presence=0.9989808201789856
Frame 0 | Landmark #2
x=0.461
y=0.565
z=-1.732
visibility=0.9991770386695862
presence=0.9987483024597168
Frame 0 | Landmark #3
x=0.486
y=0.567
z=-1.731
visibility=0.999071478843689
presence=0.9983447790145874
Frame 0 | Landmark #4
x=0.345
y=0.555
z=-1.845
visibility=0.9993681311607361
presence=0.9992930889129639
Frame 0 | Landmark #5
x=0.304
y=0.551
z=-1.846
visibility=0.9993483424186707
presence=0.9993194341659546
Frame 0 | Landmark #6
x=0.266
y=0.546
z=-1.846
visibility=0.9993675351142883
presence=0.9992745518684387
Frame 0 | Landmark #7
x=0.505
y=0.582
z=-0.744
visibility=0.9982261061668396
presence=0.9978456497192383
Frame 0 | Landmark #8
x=0.192
y=0.551
z=-1.246
visibility=0.9993707537651062
presence=0.9992493987083435
Frame 0 | Landmark #9
x=0.415
y=0.664
z=-1.529
visibility=

In [71]:
df = pd.DataFrame(extracted_data)
df


,frame,part,part_idx,x,y,z,visibility,presence
0,0,head,0,0.060344,-0.604137,-0.257586,0.999154,0.999230
1,0,left shoulder,11,0.105495,-0.448331,-0.045455,0.987015,0.956319
2,0,right shoulder,12,-0.149624,-0.435491,-0.159053,0.990897,0.977459
3,0,left elbow,13,0.087928,-0.321413,-0.047882,0.177682,0.066476
4,0,right elbow,14,-0.158428,-0.195147,-0.153220,0.179995,0.015819
...,...,...,...,...,...,...,...,...
2166,181,right hip,24,-0.096994,0.004038,-0.055607,0.013594,0.000158
2167,181,left knee,25,0.129641,-0.123378,-0.008920,0.016743,0.000142
2168,181,right knee,26,-0.093834,0.009469,-0.173397,0.046887,0.000198
2169,181,left ankle,27,0.131347,0.150201,0.257683,0.006132,0.000080


### Xu Ly data balance

In [72]:
df_balance = df[df['part'].isin(['left hip', 'right hip', 'left shoulder', 'right shoulder'])]
df_balance.to_csv('balance_data.csv', index=False)
df_balance

,frame,part,part_idx,x,y,z,visibility,presence
1,0,left shoulder,11,0.105495,-0.448331,-0.045455,0.987015,0.956319
2,0,right shoulder,12,-0.149624,-0.435491,-0.159053,0.990897,0.977459
7,0,left hip,23,0.098934,-0.003050,0.062111,0.001587,0.000279
8,0,right hip,24,-0.096827,-0.007484,-0.058086,0.001987,0.000335
14,1,left shoulder,11,0.105648,-0.448475,-0.042521,0.987811,0.984075
...,...,...,...,...,...,...,...,...
2153,180,right hip,24,-0.096935,0.003585,-0.056110,0.014911,0.000166
2159,181,left shoulder,11,0.109122,-0.456020,-0.036441,0.991785,0.985711
2160,181,right shoulder,12,-0.163564,-0.438993,-0.167508,0.996505,0.994993
2165,181,left hip,23,0.097829,-0.014296,0.058917,0.013105,0.000112


In [73]:
df_balance_pivot = df_balance.pivot(index='frame', columns='part', values=['x', 'y', 'z'])
df_balance_pivot.columns = [f"{axis}_{part.replace(' ', '_')}" for axis, part in df_balance_pivot.columns]
df_balance_pivot = df_balance_pivot.reset_index()
#df_balance_pivot

In [74]:
shoulder_width_raw = np.sqrt((df_balance_pivot['x_left_shoulder'] - df_balance_pivot['x_right_shoulder'])**2 + (df_balance_pivot['y_left_shoulder'] - df_balance_pivot['y_right_shoulder'])**2 + (df_balance_pivot['z_left_shoulder'] - df_balance_pivot['z_right_shoulder'])**2)
shoulder_width = shoulder_width_raw.mean()
print(shoulder_width)

0.26302747731062087


# Tinh sway index

In [75]:
df_balance_pivot['com_x'] = (df_balance_pivot['x_left_hip'] + df_balance_pivot['x_right_hip'] ) / 2
df_balance_pivot['com_y'] = (df_balance_pivot['y_left_hip'] + df_balance_pivot['y_right_hip'] ) / 2
df_balance_pivot['com_z'] = (df_balance_pivot['z_left_hip'] + df_balance_pivot['z_right_hip'] ) / 2
df_balance_pivot['std_com_x'] = df_balance_pivot['com_x']/shoulder_width
df_balance_pivot['std_com_y'] = df_balance_pivot['com_y']/shoulder_width
df_balance_pivot['std_com_z'] = df_balance_pivot['com_z']/shoulder_width
window_size = 30
sway_x = df_balance_pivot['std_com_x'].rolling(window=window_size).std()
sway_y = df_balance_pivot['std_com_y'].rolling(window=window_size).std()
sway_z = df_balance_pivot['std_com_z'].rolling(window=window_size).std()
df_balance_pivot['sway_index'] = np.sqrt(sway_x**2 + sway_y**2 + sway_z**2).fillna(0)
#print(df_balance_pivot[['frame', 'std_com_x', 'std_com_y', 'std_com_z', 'sway_index']])

# Tinh Posture Deviation

In [76]:
mid_shoulder_x = (df_balance_pivot['x_left_shoulder'] + df_balance_pivot['x_right_shoulder']) / 2
mid_shoulder_y = (df_balance_pivot['y_left_shoulder'] + df_balance_pivot['y_right_shoulder']) / 2
mid_shoulder_z = (df_balance_pivot['z_left_shoulder'] + df_balance_pivot['z_right_shoulder']) / 2
v_x = df_balance_pivot['com_x'] - mid_shoulder_x
v_y = df_balance_pivot['com_y'] - mid_shoulder_y
v_z = df_balance_pivot['com_z'] - mid_shoulder_z
length_v = np.sqrt(v_x**2 + v_y**2 + v_z**2)
cos = v_y / length_v
cos = np.clip(cos, -1, 1)
df_balance_pivot['Posture_Deviatation'] = np.degrees(np.arccos(cos))


# Tinh CoM Movement

In [77]:
anchor_x = df_balance_pivot['std_com_x'].iloc[window_size].mean()
anchor_y = df_balance_pivot['std_com_y'].iloc[window_size].mean()
anchor_z = df_balance_pivot['std_com_z'].iloc[window_size].mean()
df_balance_pivot["CoM_Distance"] = np.sqrt((df_balance_pivot['std_com_x'] - anchor_x)**2 + (df_balance_pivot['std_com_y'] - anchor_y)**2 + (df_balance_pivot['std_com_z'] - anchor_z)**2)
df_balance_statistics = df_balance_pivot[['frame', 'sway_index', 'Posture_Deviatation', 'CoM_Distance']]
df_balance_statistics.to_csv('balance_statistics.csv', index=False)